# Exp 3 debug — where is the shared bottleneck of label propagation? (T11)

Exp 3's absolute J&F is low for every upsampler (T05: bilinear 0.127, AnyUp2D 0.154, AnyUp3D 0.231). This notebook tests, **one change at a time**, the steps that all conditions share. Each test prints numbers and draws figures. Nothing in `anyup/` or the Exp 3 notebook is changed.

| Cell | Test | What changes vs Exp 3 | Question |
|---|---|---|---|
| 4 | **A · resolution ceiling** | no model: GT masks → grid → native | Best possible J&F on a 56² grid if propagation were perfect? Do objects vanish at 56²? |
| 5 | **Baseline + B · local window** | propagation may only match within a square window | Is unconstrained matching (anywhere in the frame) causing the noise? |
| 6 | **C · DINOv2 backbone** | Swin-T 7×7 → DINOv2-base 16² / 32² | Is the backbone the bottleneck? |
| 7 | **D · Swin-T at 448** | Swin-T 7×7 → 14×14 | Is the coarse stride-32 grid the bottleneck? |
| 8 | Summary | — | one table + bar chart, saved to Drive |

Everything else stays as in Exp 3 (56² propagation, τ 0.07, top-7, frame 0 + last 6, official J&F). C and D also run the window from B, so the combinations come for free.

**Run:** T4 is enough. Run all cells in order; any test can be switched off in Cell 2 (`RUN_A` … `RUN_D`), but C and D need the baseline from Cell 5. A test that runs out of GPU memory is reported as SKIPPED with the reason, and the notebook continues. Rough time on a T4 with 30 clips and 2 conditions (estimate, not measured): A ≈ 5–10 min (CPU), Baseline + B ≈ 8 min, C ≈ 15 min, D ≈ 10 min. Use `N_CLIPS = 5` for a quick look first. **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-in-action-test"   # T11 (change if you create a T11 branch)
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
N_CLIPS      = None    # None = all 30 val clips; int = first N (quick look, e.g. 5). Time ∝ N_CLIPS. In RUN_NAME
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # as Exp 3 (checkpoint choice S06 open)
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T11/exp3_debug"
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0

CONDITIONS   = ["bilinear", "anyup3d"]   # add "anyup2d" for all three (≈ +50 % time in Cells 5–7)
SWIN_NORM    = "kinetics"                # as Exp 3

# ── protocol values shared by every test (= Exp 3) ──
PROP_SIZE    = 56      # ↓ propagation grid = upsampler output = guide size. Affinity per frame = PROP_SIZE² × 7·PROP_SIZE² floats
                       #   (56: ≈ 275 MB). Dependent: RADII / WINDOW_CD (in grid cells), Q_CHUNK, clip features on GPU (N·768·PROP_SIZE²·4 B ≤ 1 GB)
TAU, TOP_K   = 0.07, 7 # propagation temperature / neighbours (Exp 3)
N_LAST       = 6       # ↓ context frames besides frame 0 → affinity memory. Dependent: PROP_SIZE
CHUNK_SIZE   = 16      # ↓ frames per Swin-T 224 / AnyUp3D call (Exp 3). Dependent: Q_CHUNK, PROP_SIZE
Q_CHUNK      = None    # ↓ AnyUp3D query chunk (None as Exp 3 at 56²). Dependent: CHUNK_SIZE, CHUNK_SIZE_C, CHUNK_SIZE_D, PROP_SIZE
T05_REF      = {"bilinear": 0.1272, "anyup2d": 0.1543, "anyup3d": 0.2308}   # Exp 3 official J&F, T05 full run (baseline check)

# ── Test A · resolution ceiling (CPU, no model) ──
RUN_A        = True
GRIDS_A      = [(28, 28), (56, 56), (112, 112), (56, 100)]   # ↓ (h, w) grids; each = one J&F eval of all GT frames (CPU time ∝ len).
                                                             #   (56, 100) keeps the 480p aspect ratio instead of squashing to a square

# ── Baseline + Test B · local window in the propagation ──
RUN_B        = True
RADII        = [6, 12] # square window half-size in PROP_SIZE grid cells (DINO: 12 on its stride-8 grid). Propagation only (features shared)

# ── Test C · DINOv2-base per frame instead of Swin-T ──
RUN_C        = True
DINO_INPUTS  = [224, 448]  # ↓ backbone input sizes (multiples of 14) → grids 16², 32²; each = one more backbone pass over all clips.
                           #   Dependent: CHUNK_SIZE_C (448 = 4× the tokens of 224)
CHUNK_SIZE_C = 16      # ↓ frames per DINOv2 / AnyUp3D call. Dependent: DINO_INPUTS (reduce to 8 if 448 runs out of memory)

# ── Test D · Swin-T at a larger input (finer grid) ──
RUN_D        = True
SWIN_INPUT_D = 448     # ↓ Swin-T input (multiple of 32 → 14×14 grid at 448); ≈ 4× the memory of 224. Dependent: CHUNK_SIZE_D
CHUNK_SIZE_D = 8       # ↓ frames per Swin-T / AnyUp3D call at SWIN_INPUT_D. Dependent: SWIN_INPUT_D

WINDOW_CD    = 12      # window radius also run in C and D (combination with B); None = no window there

# ── figures ──
N_SHOW       = 4       # rows of each mask figure: the clips with the largest gains + the largest loss
SHOW_FRAC    = 0.8     # mask figures show the frame at this fraction of the clip (propagation errors grow over time)
SHOW_H       = 200     # panel height in px (width keeps the aspect ratio); display only
SHOW_CLIP    = None    # clip of the feature-grid (PCA) figures; None = first clip

In [ ]:
# ── Cell 3 · Imports, data, models, helpers ────────────────────────────────
import time, json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from anyup import davis, metrics, viz
from anyup.backbones import BackboneLoader
from anyup.video import iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader
from anyup.utils.seed import set_seed   # shared: seeds + deterministic cuDNN / algorithms (T07)

set_seed(SEED)
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
ROOT  = davis.download_davis(DAVIS_DEST)
CLIPS = davis.list_clips(ROOT, "val")[:N_CLIPS]                            # [:None] = all
RUN_NAME = f"p{PROP_SIZE}_n{len(CLIPS)}_{'-'.join(CONDITIONS)}"
OUT_DIR  = Path(OUTPUT_ROOT) / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
SHOW_CLIP = SHOW_CLIP or CLIPS[0]
print(f"{len(CLIPS)} clips | RUN_NAME {RUN_NAME} | outputs → {OUT_DIR}")

UPS = {}
for cond in CONDITIONS:
    kw = dict(ckpt_path=CKPT_ANYUP3D, q_chunk_size=Q_CHUNK) if cond == "anyup3d" else {}
    UPS[cond] = UpsamplerLoader.load(cond, device=DEVICE, **kw)
print("upsamplers:", list(UPS))

RES, PREDS, GRIDS, SKIPPED = {}, {}, {}, {}   # variant → cond → clip → eval result | (T, P, P) uint8 preds; variant → feature grid; test → reason
LR_PANELS = {}                                 # backbone name → (PCA RGB of its LR features (h, w, 3), grid)


def propagate_local(feats, ann0, radius=None, tau=TAU, top_k=TOP_K, n_last=N_LAST):
    """
    davis.propagate_labels (same lines, same order) + an optional square window: a query pixel may only
    take labels from context pixels within `radius` grid cells (Chebyshev distance), as DINO's
    size_mask_neighborhood. radius=None → identical to davis.propagate_labels (checked in Cell 5).
    feats (T, C, H, W) on any device, frame 0 = annotated; ann0 (H0, W0) GT of frame 0 → (T, H, W) int64 CPU.
    """
    T, C, H, W = feats.shape
    N = H * W
    feats_flat = F.normalize(feats.reshape(T, C, N).permute(0, 2, 1), dim=-1)   # (T, N, C) unit vectors
    ann0 = np.asarray(ann0)
    if (ann0 == 255).any():
        raise ValueError("ann0 contains void (255) pixels")
    n_objects = int(ann0.max())
    n_ch = n_objects + 1
    label_maps = torch.zeros(T, n_ch, N, device=feats.device)                     # (T, n_ch, N)
    label_maps[0] = davis._label_maps_frame0(ann0, n_objects, H, W).to(feats.device)
    near, near_rep = None, {}
    if radius is not None:
        yy, xx = torch.meshgrid(torch.arange(H, device=feats.device), torch.arange(W, device=feats.device), indexing="ij")
        pos = torch.stack([yy.flatten(), xx.flatten()], dim=1)                     # (N, 2) grid position per pixel
        near = (pos[:, None, :] - pos[None, :, :]).abs().amax(dim=-1) <= radius    # (N, N) bool, square window
    for t in range(1, T):
        past = list(range(max(1, t - n_last), t))
        context_t = [0] + past                                                     # frame 0 always first
        ctx_feats = feats_flat[context_t].reshape(-1, C)                           # (Nctx·N, C), frame-major
        ctx_labels = label_maps[context_t].permute(1, 0, 2).reshape(n_ch, -1)      # (n_ch, Nctx·N)
        A = torch.mm(feats_flat[t], ctx_feats.T) / tau                             # (N, Nctx·N)
        if near is not None:
            if len(context_t) not in near_rep:
                near_rep[len(context_t)] = near.repeat(1, len(context_t))          # (N, Nctx·N), same frame-major order
            A = A.masked_fill(~near_rep[len(context_t)], float("-inf"))
        topk_vals, topk_idx = A.topk(top_k, dim=-1)                                # (N, top_k)
        A_sparse = torch.full_like(A, float("-inf"))
        A_sparse.scatter_(1, topk_idx, topk_vals)
        A_soft = F.softmax(A_sparse, dim=-1)                                       # (N, Nctx·N)
        label_maps[t] = torch.mm(A_soft, ctx_labels.T).T                           # (n_ch, N)
    return label_maps.argmax(dim=1).view(T, H, W).cpu()                           # (T, H, W)


def score(pred_lr, gt):
    """pred_lr (T, h, w) ids → native (nearest, the cv2 call of save_mask_png) → official J&F of one clip."""
    pred = np.stack([davis.resize_labels(np.asarray(p, np.uint8), gt.shape[1:]) for p in pred_lr])   # (T, H0, W0)
    return metrics.eval_clip_jf(gt, pred)


def clip_mean_jf(res):
    j, f = float(np.mean(res["J_M"])), float(np.mean(res["F_M"]))
    return (j + f) / 2


def run_backbone(bb, tag, chunk_size, radii):
    """
    Backbone once per clip → per condition: upsample every chunk to PROP_SIZE, propagate with each radius, score.
    Variants: `tag` (no window) and `tag+w<r>`. Fills RES / PREDS / GRIDS. Returns the variant names.
    On CUDA out-of-memory: removes the partial variants, records the reason in SKIPPED and returns [].
    """
    names = {r: tag if r is None else f"{tag}+w{r}" for r in radii}
    for n in names.values():
        RES[n] = {c: {} for c in CONDITIONS}
        PREDS[n] = {c: {} for c in CONDITIONS}
    t_start = time.perf_counter()
    try:
        for ci, clip in enumerate(CLIPS):
            t0 = time.perf_counter()
            gt = davis.load_annotations(ROOT, clip)                                 # (T, 480, W) uint8, native
            chunks = list(iter_backbone_chunks(bb, davis.open_clip(ROOT, clip), chunk_size=chunk_size, overlap=0))
            assert [f for ch in chunks for f in ch.meta["frame_index"]] == list(range(len(gt))), "frame mismatch"
            for n in names.values():
                GRIDS[n] = tuple(chunks[0].meta["grid"])
            line = []
            for cond in CONDITIONS:
                feats = torch.cat([UPS[cond](ch, output_size=PROP_SIZE).features for ch in chunks])   # (T, C, P, P) GPU
                for r, n in names.items():
                    pred = propagate_local(feats, gt[0], radius=r)                  # (T, P, P) CPU
                    PREDS[n][cond][clip] = pred.numpy().astype(np.uint8)
                    RES[n][cond][clip] = score(pred.numpy(), gt)
                    line.append(f"{n}/{cond} {clip_mean_jf(RES[n][cond][clip]):.3f}")
                del feats
            del chunks
            torch.cuda.empty_cache()
            print(f"[{ci + 1}/{len(CLIPS)}] {clip:<18} " + "  ".join(line) + f"  {time.perf_counter() - t0:.0f}s")
    except torch.cuda.OutOfMemoryError as e:
        for n in names.values():
            RES.pop(n, None); PREDS.pop(n, None); GRIDS.pop(n, None)
        torch.cuda.empty_cache()
        SKIPPED[tag] = f"CUDA out of memory on {GPU_NAME} ({str(e).splitlines()[0][:120]}); reduce the chunk size of this test"
        print(f"SKIPPED {tag}: {SKIPPED[tag]}")
        return []
    print(f"{tag}: feature grid {GRIDS[names[radii[0]]]}, {time.perf_counter() - t_start:.0f}s")
    return list(names.values())


def official(variant, cond):
    return metrics.summarize_jf(RES[variant][cond])


def per_clip(variant, cond):
    return {c: v["JF"] for c, v in official(variant, cond)["per_clip"].items()}


def print_table(variants, base=None):
    print(f"{'variant':<16} {'cond':<9} {'grid':>9} {'J&F':>7} {'J':>7} {'F':>7}" + ("   Δ J&F vs base   better/clips" if base else ""))
    for v in variants:
        for c in CONDITIONS:
            s = official(v, c)
            row = f"{v:<16} {c:<9} {str(GRIDS[v]):>9} {s['J&F-Mean']:>7.4f} {s['J-Mean']:>7.4f} {s['F-Mean']:>7.4f}"
            if base and v != base:
                a, b = per_clip(base, c), per_clip(v, c)
                up = sum(b[k] > a[k] for k in a)
                row += f"   {s['J&F-Mean'] - official(base, c)['J&F-Mean']:+.4f}        {up}/{len(a)}"
            print(row)


def scatter_figure(base, others, title, path):
    """Per-clip J&F: base (x) vs each other variant (y), one panel per condition. Above the diagonal = variant better."""
    fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(5.4 * len(CONDITIONS), 5.2), squeeze=False)
    for ax, cond in zip(axes[0], CONDITIONS):
        a = per_clip(base, cond)
        clips = list(a)
        ax.plot([0, 1], [0, 1], "k--", lw=0.8)
        for v in others:
            if v not in RES:
                continue
            b = per_clip(v, cond)
            up = sum(b[k] > a[k] for k in clips)
            ax.scatter([a[k] for k in clips], [b[k] for k in clips], s=20, alpha=0.8,
                       label=f"{v}: J&F {official(v, cond)['J&F-Mean']:.3f}, better on {up}/{len(clips)}")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal"); ax.grid(alpha=0.3)
        ax.set_xlabel(f"{base} per-clip J&F  (official {official(base, cond)['J&F-Mean']:.3f})")
        ax.set_ylabel("variant per-clip J&F")
        ax.set_title(cond, fontweight="bold")
        ax.legend(fontsize=7, loc="upper left")
    fig.suptitle(title + "  (each dot = one clip; above the dashed line = better than the baseline)", fontsize=11)
    plt.tight_layout()
    plt.savefig(path, dpi=130, bbox_inches="tight"); plt.show()


def pick_clips(base, other, cond):
    """The N_SHOW−1 clips where `other` gains most over `base` + the clip where it loses most."""
    a, b = per_clip(base, cond), per_clip(other, cond)
    d = {k: b[k] - a[k] for k in a}
    ranked = sorted(d, key=d.get, reverse=True)
    sel = ranked[:max(N_SHOW - 1, 0)]
    if ranked[-1] not in sel:
        sel.append(ranked[-1])
    return sel, d


def frame_and_gt(clip, t):
    """Native frame t + its GT, resized to height SHOW_H (aspect kept)."""
    img = np.array(Image.open(davis.frame_paths(ROOT, clip)[t]).convert("RGB"))
    hw = (SHOW_H, int(round(img.shape[1] * SHOW_H / img.shape[0])))
    gt = davis.load_annotation(davis.ann_paths(ROOT, clip)[t])
    return viz.resize_rgb(img, hw), davis.resize_labels(gt, hw), hw


def mask_figure(variants, sel, delta, title, path):
    """Rows = clips, columns = GT | each variant × condition, all at the frame SHOW_FRAC through the clip."""
    cols = [(v, c) for v in variants for c in CONDITIONS if v in RES]
    jfs = {(v, c): per_clip(v, c) for v, c in cols}
    fig, axes = plt.subplots(len(sel), 1 + len(cols), figsize=(3.4 * (1 + len(cols)), 2.3 * len(sel) + 0.8), squeeze=False)
    for r, clip in enumerate(sel):
        T = len(davis.frame_paths(ROOT, clip))
        t = min(T - 1, int(round(SHOW_FRAC * (T - 1))))
        img, gt, hw = frame_and_gt(clip, t)
        panels = [(viz.overlay(img, gt), f"{clip}  (Δ {delta[clip]:+.3f})\nground truth, frame {t}/{T - 1}")]
        for v, c in cols:
            pred = davis.resize_labels(PREDS[v][c][clip][t], hw)
            panels.append((viz.overlay(img, pred), f"{v} · {c}\nJ&F {jfs[v, c][clip]:.3f}"))
        for ax, (p, cap) in zip(axes[r], panels):
            ax.imshow(p); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(cap, fontsize=8)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(path, dpi=130, bbox_inches="tight"); plt.show()


@torch.no_grad()
def lr_panel(bb, name):
    """PCA colours of the backbone's LOW-RES features of one frame (SHOW_CLIP, middle frame) → LR_PANELS[name]."""
    fp = davis.frame_paths(ROOT, SHOW_CLIP)
    x = torch.from_numpy(np.array(Image.open(fp[len(fp) // 2]).convert("RGB"))).permute(2, 0, 1)[None].float() / 255   # (1, 3, 480, W)
    f = bb.forward(x).features                                                    # (1, C, h, w)
    pca = viz.FeaturePCA(seed=SEED).fit({"lr": viz.FeaturePCA.sample_tokens(f, 10_000, SEED)})
    LR_PANELS[name] = (pca.rgb(f, "lr")[0], tuple(f.shape[-2:]))                  # (h, w, 3) uint8


def grid_figure(names, title, path):
    """Frame with each backbone's token grid drawn on it + the PCA colours of its LR features (nearest, no smoothing)."""
    fp = davis.frame_paths(ROOT, SHOW_CLIP)
    img = np.array(Image.open(fp[len(fp) // 2]).convert("RGB"))
    H0, W0 = img.shape[:2]
    names = [n for n in names if n in LR_PANELS]
    fig, axes = plt.subplots(2, len(names), figsize=(4.6 * len(names), 5.6), squeeze=False)
    for j, n in enumerate(names):
        rgb, (h, w) = LR_PANELS[n]
        axes[0, j].imshow(img)
        for k in range(1, h):
            axes[0, j].axhline(k * H0 / h, color="w", lw=0.6)
        for k in range(1, w):
            axes[0, j].axvline(k * W0 / w, color="w", lw=0.6)
        axes[0, j].set_title(f"{n}: {h}×{w} tokens per frame", fontsize=10)
        axes[1, j].imshow(viz.resize_rgb(rgb, (H0, W0), nearest=True))
        axes[1, j].set_title("its features (PCA colours)", fontsize=9)
        for ax in axes[:, j]:
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(path, dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ── Cell 4 · Test A · resolution ceiling (no model) ────────────────────────
# GT masks of every frame → nearest to the grid → nearest back to native → official J&F. This is the score a PERFECT
# propagation would get on that grid, so it is an upper bound for Exp 3 at PROP_SIZE. Also counts frame-0 objects
# that (almost) vanish on the PROP_SIZE² grid: those can never be propagated.
if RUN_A:
    t_start = time.perf_counter()
    CEIL = {g: {} for g in GRIDS_A}
    lost = []
    for ci, clip in enumerate(CLIPS):
        gt = davis.load_annotations(ROOT, clip)                                     # (T, 480, W) uint8
        H0, W0 = gt.shape[1:]
        for g in GRIDS_A:
            rt = np.stack([davis.resize_labels(davis.resize_labels(m, g), (H0, W0)) for m in gt])   # (T, 480, W)
            rt[rt == 255] = 0                                                       # void → bg (a prediction has no void)
            CEIL[g][clip] = metrics.eval_clip_jf(gt, rt)
        a0 = davis.resize_labels(gt[0], (PROP_SIZE, PROP_SIZE))
        for k in range(1, int(gt[0][gt[0] != 255].max()) + 1):
            n_nat, n_lr = int((gt[0] == k).sum()), int((a0 == k).sum())
            if n_lr <= 2:
                lost.append((clip, k, n_nat, n_lr))
        print(f"[{ci + 1}/{len(CLIPS)}] {clip:<18} " +
              "  ".join(f"{g[0]}×{g[1]}: {clip_mean_jf(CEIL[g][clip]):.3f}" for g in GRIDS_A))
    CEIL_SUM = {g: metrics.summarize_jf(CEIL[g]) for g in GRIDS_A}
    print(f"\nTest A done in {time.perf_counter() - t_start:.0f}s. Ceiling = J&F of the GT itself after the grid round trip:")
    for g in GRIDS_A:
        s = CEIL_SUM[g]
        print(f"  grid {g[0]:>3}×{g[1]:<3}  J&F {s['J&F-Mean']:.4f}  J {s['J-Mean']:.4f}  F {s['F-Mean']:.4f}")
    print(f"  Exp 3 now (T05, 56×56): " + ", ".join(f"{c} {v:.4f}" for c, v in T05_REF.items()))
    print(f"\nframe-0 objects with ≤ 2 pixels on the {PROP_SIZE}² grid (cannot be propagated): {len(lost)}")
    for clip, k, n_nat, n_lr in lost:
        print(f"  {clip} object {k}: {n_nat} px native → {n_lr} px")

    # figure 1: ceiling per grid vs what Exp 3 reaches now
    fig, ax = plt.subplots(figsize=(10, 4.2))
    labels = [f"{g[0]}×{g[1]}" + (" (aspect kept)" if g[0] != g[1] else "") for g in GRIDS_A]
    x = np.arange(len(GRIDS_A))
    for k, (key, lab) in enumerate([("J&F-Mean", "J&F"), ("J-Mean", "J"), ("F-Mean", "F")]):
        ax.bar(x + (k - 1) * 0.25, [CEIL_SUM[g][key] for g in GRIDS_A], 0.25, label=f"ceiling {lab}",
               color=["#1f4e79", "#4f8cc9", "#a9c8ea"][k])
    for c, v in T05_REF.items():
        ax.axhline(v, ls="--", lw=1.2, color={"bilinear": "0.35", "anyup2d": "tab:orange", "anyup3d": "tab:red"}[c],
                   label=f"Exp 3 now, {c} ({v:.3f})")
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.05); ax.set_ylabel("official score")
    ax.set_title("Test A: best possible score if propagation were perfect (GT through the grid and back)")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1)); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "A_ceiling.png", dpi=130, bbox_inches="tight"); plt.show()

    # figure 2: the clips that lose most on the PROP_SIZE² grid: GT vs GT after the round trip
    g56 = (PROP_SIZE, PROP_SIZE)
    worst = sorted(CEIL[g56], key=lambda c: clip_mean_jf(CEIL[g56][c]))[:N_SHOW]
    fig, axes = plt.subplots(len(worst), 2, figsize=(9, 2.4 * len(worst) + 0.6), squeeze=False)
    for r, clip in enumerate(worst):
        T = len(davis.frame_paths(ROOT, clip))
        t = min(T - 1, int(round(SHOW_FRAC * (T - 1))))
        img, gt_s, hw = frame_and_gt(clip, t)
        gt_nat = davis.load_annotation(davis.ann_paths(ROOT, clip)[t])
        rt = davis.resize_labels(davis.resize_labels(gt_nat, g56), hw)
        axes[r, 0].imshow(viz.overlay(img, gt_s)); axes[r, 0].set_title(f"{clip}: ground truth, frame {t}", fontsize=9)
        axes[r, 1].imshow(viz.overlay(img, rt))
        axes[r, 1].set_title(f"same GT through {PROP_SIZE}² and back (clip ceiling J&F {clip_mean_jf(CEIL[g56][clip]):.3f})", fontsize=9)
        for ax in axes[r]:
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Test A: the {len(worst)} clips that lose most on the {PROP_SIZE}² grid", fontsize=11)
    plt.tight_layout(); plt.savefig(OUT_DIR / "A_worst_clips.png", dpi=130, bbox_inches="tight"); plt.show()
else:
    print("RUN_A=False — skipped")

In [ ]:
# ── Cell 5 · Baseline (= Exp 3) + Test B · local window ────────────────────
# Swin-T 224 (7×7 grid) as in Exp 3. The same upsampled features are propagated without a window (= Exp 3) and with
# square windows of RADII grid cells, so only the matching rule changes.
if RUN_B:
    bb = BackboneLoader.load("swin_t", device=DEVICE, guide_size=PROP_SIZE, temporal_mode="per_frame", norm=SWIN_NORM)

    # check (i): the notebook's propagation without a window = davis.propagate_labels (first clip, first condition)
    gt0 = davis.load_annotations(ROOT, CLIPS[0])
    ch0 = list(iter_backbone_chunks(bb, davis.open_clip(ROOT, CLIPS[0]), chunk_size=CHUNK_SIZE, overlap=0))
    f0 = torch.cat([UPS[CONDITIONS[0]](ch, output_size=PROP_SIZE).features for ch in ch0])   # (T, 768, P, P)
    n_diff = int((propagate_local(f0, gt0[0]) != davis.propagate_labels(f0, gt0[0], tau=TAU, top_k=TOP_K, n_last=N_LAST)).sum())
    print(f"check (i) propagate_local(radius=None) vs davis.propagate_labels on {CLIPS[0]}: {n_diff} differing pixels "
          f"→ {'PASS' if n_diff == 0 else 'FAIL'}")
    del ch0, f0
    torch.cuda.empty_cache()

    lr_panel(bb, "swin_t 224")
    B_NAMES = run_backbone(bb, "swin224", CHUNK_SIZE, [None] + list(RADII))
    del bb
    torch.cuda.empty_cache()

    if "swin224" not in RES:
        raise RuntimeError(f"baseline skipped ({SKIPPED.get('swin224')}); reduce CHUNK_SIZE and re-run this cell")

    # check (ii): the baseline reproduces the T05 Exp 3 numbers (only meaningful with all 30 clips)
    for c in CONDITIONS:
        v = official("swin224", c)["J&F-Mean"]
        if len(CLIPS) == 30 and c in T05_REF:
            ok = abs(v - T05_REF[c]) < 0.002
            print(f"check (ii) baseline {c}: J&F {v:.4f} vs T05 {T05_REF[c]:.4f} → {'PASS' if ok else 'WARN (|diff| ≥ 0.002)'}")
        else:
            print(f"check (ii) baseline {c}: J&F {v:.4f} (not compared: needs all 30 clips)")

    print()
    print_table(B_NAMES, base="swin224")
    if len(B_NAMES) > 1:
        best_b = max(B_NAMES[1:], key=lambda v: official(v, CONDITIONS[0])["J&F-Mean"])
        scatter_figure("swin224", B_NAMES[1:], "Test B: local window vs no window (Swin-T 224)", OUT_DIR / "B_scatter.png")
        sel, d = pick_clips("swin224", best_b, CONDITIONS[0])
        mask_figure(["swin224", best_b], sel, d,
                    f"Test B: no window vs {best_b}  (rows: largest gains + largest loss for {CONDITIONS[0]})", OUT_DIR / "B_masks.png")
else:
    print("RUN_B=False — skipped (Tests C and D need this baseline)")

In [ ]:
# ── Cell 6 · Test C · DINOv2-base per frame instead of Swin-T ──────────────
# Same propagation, upsamplers, sizes and scoring; only the backbone changes (DINOv2 is self-supervised, made for
# dense features; Swin-T is a supervised action classifier). Without and with the window (WINDOW_CD).
if RUN_C:
    assert "swin224" in RES, "Test C compares against the baseline: run Cell 5 first (RUN_B=True)"
    C_NAMES = []
    for size in DINO_INPUTS:
        bb = BackboneLoader.load("dinov2", device=DEVICE, input_size=size, guide_size=PROP_SIZE, variant="base")
        lr_panel(bb, f"dinov2 {size}")
        C_NAMES += run_backbone(bb, f"dino{size}", CHUNK_SIZE_C, [None] + ([WINDOW_CD] if WINDOW_CD else []))
        del bb
        torch.cuda.empty_cache()
    if C_NAMES:
        print()
        print_table(["swin224"] + C_NAMES, base="swin224")
        scatter_figure("swin224", C_NAMES, "Test C: DINOv2 vs Swin-T (baseline)", OUT_DIR / "C_scatter.png")
        best_c = max(C_NAMES, key=lambda v: official(v, CONDITIONS[0])["J&F-Mean"])
        sel, d = pick_clips("swin224", best_c, CONDITIONS[0])
        mask_figure(["swin224", best_c], sel, d, f"Test C: Swin-T baseline vs {best_c}", OUT_DIR / "C_masks.png")
    grid_figure(["swin_t 224"] + [f"dinov2 {s}" for s in DINO_INPUTS],
                f"Test C: how fine each backbone sees the frame ({SHOW_CLIP})", OUT_DIR / "C_grids.png")
else:
    print("RUN_C=False — skipped")

In [ ]:
# ── Cell 7 · Test D · Swin-T at a larger input (finer grid) ────────────────
# Same backbone as Exp 3, input SWIN_INPUT_D instead of 224 → a 2× finer grid (14×14 at 448). Isolates the grid size
# from the backbone type (compare with Test C).
if RUN_D:
    assert "swin224" in RES, "Test D compares against the baseline: run Cell 5 first (RUN_B=True)"
    bb = BackboneLoader.load("swin_t", device=DEVICE, input_size=SWIN_INPUT_D, guide_size=PROP_SIZE,
                             temporal_mode="per_frame", norm=SWIN_NORM)
    lr_panel(bb, f"swin_t {SWIN_INPUT_D}")
    D_NAMES = run_backbone(bb, f"swin{SWIN_INPUT_D}", CHUNK_SIZE_D, [None] + ([WINDOW_CD] if WINDOW_CD else []))
    del bb
    torch.cuda.empty_cache()
    if D_NAMES:
        print()
        print_table(["swin224"] + D_NAMES, base="swin224")
        scatter_figure("swin224", D_NAMES, f"Test D: Swin-T {SWIN_INPUT_D} vs Swin-T 224 (baseline)", OUT_DIR / "D_scatter.png")
        best_d = max(D_NAMES, key=lambda v: official(v, CONDITIONS[0])["J&F-Mean"])
        sel, d = pick_clips("swin224", best_d, CONDITIONS[0])
        mask_figure(["swin224", best_d], sel, d, f"Test D: Swin-T 224 vs {best_d}", OUT_DIR / "D_masks.png")
    grid_figure(["swin_t 224", f"swin_t {SWIN_INPUT_D}"], f"Test D: Swin-T grid at 224 vs {SWIN_INPUT_D} ({SHOW_CLIP})",
                OUT_DIR / "D_grids.png")
else:
    print("RUN_D=False — skipped")

In [ ]:
# ── Cell 8 · Summary + save ────────────────────────────────────────────────
variants = list(RES)
base = "swin224" if "swin224" in RES else None
if variants:
    print_table(variants, base=base)
if "CEIL_SUM" in globals():
    print("\nTest A ceilings (perfect propagation): " +
          ", ".join(f"{g[0]}×{g[1]} {CEIL_SUM[g]['J&F-Mean']:.3f}" for g in GRIDS_A))
for k, why in SKIPPED.items():
    print(f"SKIPPED {k}: {why}")

rows, summary = [], {"variants": {}, "paired_vs_baseline": {}, "skipped": SKIPPED, "grids": GRIDS}
for v in variants:
    for c in CONDITIONS:
        s = official(v, c)
        summary["variants"][f"{v}/{c}"] = {k: s[k] for k in ("J&F-Mean", "J-Mean", "J-Recall", "J-Decay", "F-Mean",
                                                             "F-Recall", "F-Decay", "n_clips", "n_objects")}
        for clip, pc in s["per_clip"].items():
            rows.append(dict(variant=v, condition=c, clip=clip, grid=str(GRIDS[v]), J=pc["J"], F=pc["F"], JF=pc["JF"],
                             n_objects=pc["n_objects"]))
        if base and v != base:
            a, b = per_clip(base, c), per_clip(v, c)
            summary["paired_vs_baseline"][f"{v}/{c}"] = metrics.paired_stats([b[k] for k in a], [a[k] for k in a])
if "CEIL_SUM" in globals():
    summary["ceiling"] = {f"{g[0]}x{g[1]}": {k: CEIL_SUM[g][k] for k in ("J&F-Mean", "J-Mean", "F-Mean")} for g in GRIDS_A}

if variants:
    fig, ax = plt.subplots(figsize=(max(8, 1.3 * len(variants)), 4.5))
    x = np.arange(len(variants))
    wdt = 0.8 / len(CONDITIONS)
    for k, c in enumerate(CONDITIONS):
        vals = [official(v, c)["J&F-Mean"] for v in variants]
        bars = ax.bar(x + (k - (len(CONDITIONS) - 1) / 2) * wdt, vals, wdt, label=c)
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
    if "CEIL_SUM" in globals():
        ax.axhline(CEIL_SUM[(PROP_SIZE, PROP_SIZE)]["J&F-Mean"], color="k", ls="--", lw=1,
                   label=f"ceiling at {PROP_SIZE}² (perfect propagation)")
    ax.set_xticks(x); ax.set_xticklabels([f"{v}\n{GRIDS[v][0]}×{GRIDS[v][1]}" for v in variants], fontsize=8)
    ax.set_ylabel("official J&F"); ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
    ax.set_title("Exp 3 debug: every variant (baseline = swin224 = Exp 3)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "summary.png", dpi=130, bbox_inches="tight"); plt.show()

config = dict(N_CLIPS=N_CLIPS, clips=CLIPS, CKPT_ANYUP3D=CKPT_ANYUP3D, SEED=SEED, CONDITIONS=CONDITIONS, SWIN_NORM=SWIN_NORM,
              PROP_SIZE=PROP_SIZE, TAU=TAU, TOP_K=TOP_K, N_LAST=N_LAST, CHUNK_SIZE=CHUNK_SIZE, Q_CHUNK=Q_CHUNK,
              GRIDS_A=GRIDS_A, RADII=RADII, DINO_INPUTS=DINO_INPUTS, CHUNK_SIZE_C=CHUNK_SIZE_C, SWIN_INPUT_D=SWIN_INPUT_D,
              CHUNK_SIZE_D=CHUNK_SIZE_D, WINDOW_CD=WINDOW_CD, RUN=dict(A=RUN_A, B=RUN_B, C=RUN_C, D=RUN_D),
              gpu=GPU_NAME, repo_branch=REPO_BRANCH)
csv_path, json_path = metrics.save_results(OUT_DIR, rows, summary, config, name="exp3_debug")
print("\nsaved:", csv_path, json_path, "+ figures in", OUT_DIR)